# Lab 0-04: Memory Gives an Agent Useful Context

An agent can carry selected information from one step to a later step. This is **memory**: context that the agent deliberately includes in a later model request. Memory is not a new capability inside the `LLM`, and it does not create new evidence.

In this tutorial, you will see two uses of memory:

1. **Conversation memory:** carrying a user's name from an earlier exchange into a follow-up question.
2. **Evidence-bounded working memory:** carrying observed facts and known limits into a cautious forensic summary.

Both examples use the Qwen/Ollama connection configured for this lab.

## 1. What Is Agent Memory?

Agent memory is information that the surrounding workflow saves and later supplies as context to the `LLM`. It does **not** permanently change the model's trained parameters.

Two useful categories are:

- **Short-term memory:** information kept during one conversation or task, such as a user's name, prior messages, intermediate results, or current evidence notes.
- **Long-term memory:** information kept across separate sessions, usually in an external store such as a file, database, or vector store, then retrieved when it is relevant.

Memory must be selected, limited, and reviewed. This matters especially in forensic work: saved notes should preserve what was observed and what remains unknown, rather than becoming unverified evidence.

This tutorial demonstrates **short-term memory only**. The first example carries a person's name into a later request. The second carries bounded observations, unknowns, and a next human-review step during one synthetic case review. Long-term memory is introduced here as a concept; this notebook does not create persistent storage or retrieval.

## 2. Connect to the Configured Qwen Model

This setup reads the model name and endpoint from this lab's `.env` file. It also defines one small helper that sends a chosen list of messages to Qwen. The important idea is that **the agent chooses which earlier information to include in that list**.

In [ ]:
# Import only the libraries needed to locate settings, save session memory, and contact the local model.
import json

from pathlib import Path

from dotenv import dotenv_values
from openai import OpenAI

# This check prevents the notebook from silently reading the wrong lab's files.
lab_dir = Path.cwd().resolve()
if lab_dir.name != "lab0_04_ai_agent":
    raise RuntimeError(
        "Run this notebook from the lab0_04_ai_agent folder so it can find the lab-local .env file."
    )

# Load the model settings that the student configured for this lab.
settings = dotenv_values(lab_dir / ".env")
model_name = settings.get("MODEL")
base_url = settings.get("OLLAMA_BASE_URL")
if not model_name or not base_url:
    raise RuntimeError(
        "Missing MODEL or OLLAMA_BASE_URL. Copy .env.example to .env and complete Lab 0-02 first."
    )

# Create an OpenAI-compatible client for the local Ollama server.
client = OpenAI(base_url=base_url, api_key="ollama")
print(f"Using {model_name} at {base_url}")


def ask_qwen(messages: list[dict[str, str]]) -> str:
    """Send the selected context to Qwen and return its text response."""
    # The messages list is the complete context Qwen can use for this one request.
    response = client.chat.completions.create(
        model=model_name,
        messages=messages,
        temperature=0,
    )
    answer = response.choices[0].message.content
    if not answer:
        raise RuntimeError("Qwen returned an empty response. Check that Ollama is running and try again.")
    return answer.strip()


## 3. Part 1: Conversation Memory

A bare model call does not automatically remember a previous call. To make a follow-up answer use a person's name from an earlier exchange, the agent must add that name to the next request's context.

The first request below includes no name. The second writes the name to a temporary session file, reloads it, and includes that short-term memory in the next request. The file is removed after the demonstration.

In [ ]:
# Imagine that the user introduced herself earlier in a conversation.
# This temporary external file makes the agent's short-term memory visible to students.
session_path = lab_dir / ".memory_demo_session.json"
session_memory = {"user_name": "Maya"}
session_path.write_text(json.dumps(session_memory, indent=2), encoding="utf-8")
print(f"Temporary session file created: {session_path.name}")
print(session_path.read_text(encoding="utf-8"))

# A later agent step reloads the selected information from the same temporary session.
loaded_session = json.loads(session_path.read_text(encoding="utf-8"))
person_name = loaded_session["user_name"]
follow_up_question = "What is my name?"

# This request omits the earlier name, so the model should not claim to know it.
without_memory = ask_qwen(
    [
        {
            "role": "system",
            "content": (
                "No name is provided in this request. "
                "Reply exactly: I do not know your name."
            ),
        },
        {"role": "user", "content": follow_up_question},
    ]
)

# This request adds the earlier name as agent-managed conversation memory.
with_memory = ask_qwen(
    [
        {
            "role": "system",
            "content": "Answer the user's question using the supplied conversation memory.",
        },
        {
            "role": "system",
            "content": f"Conversation memory from an earlier turn: The user's name is {person_name}.",
        },
        {"role": "user", "content": follow_up_question},
    ]
)

print("Without memory:\n", without_memory)
print("\nWith memory reloaded from the temporary session file:\n", with_memory)

# Remove the file so this remains a short-term-memory demonstration, not persistent storage.
session_path.unlink(missing_ok=True)
print(f"\nTemporary session file removed: {not session_path.exists()}")


The model's parameters did not change between these calls. Only the **context supplied by the agent** changed. In a longer workflow, an agent might keep a short, relevant record of user preferences, completed steps, or intermediate results rather than repeatedly sending an entire conversation.

## 4. Part 2: Evidence-Bounded Forensic Working Memory

In a forensic workflow, memory should be even more deliberate. It can retain a compact record of observed artifacts, questions already answered, and important limits. It must not turn an inference into a fact or fill gaps with plausible-sounding details.

The working memory below summarizes the approved **synthetic** case packet. It records observations separately from what the packet does not establish.

In [ ]:
# This is a deliberately small working note derived from the approved synthetic case packet.
# Keeping observations and unknowns separate helps prevent the agent from overstating the evidence.
forensic_memory = {
    "observations": [
        "A staff-schedule screenshot was created, a Messages conversation was opened, a message said 'sending that image now', an outgoing MMS connection occurred, and the screenshot was later deleted."
    ],
    "unknowns": [
        "The packet does not establish the screenshot's contents, the recipient's identity, or whether the image was successfully delivered."
    ],
    "next_human_review": [
        "Review the original message and media artifacts, if available, before reaching a stronger conclusion."
    ],
}

review_question = (
    "Did the records prove that the staff-schedule screenshot was successfully delivered? "
    "Give a cautious answer and one next human-review step."
)

# The agent supplies its bounded working memory as context for this specific review step.
forensic_answer = ask_qwen(
    [
        {
            "role": "system",
            "content": (
                "You are an evidence-bounded mobile-device activity summary assistant. "
                "Use only the supplied working memory. Do not claim that an unknown fact is established. "
                "Respond with three short labeled parts: Observed, Still unknown, and Next human-review step."
            ),
        },
        {"role": "system", "content": f"Working memory: {forensic_memory}"},
        {"role": "user", "content": review_question},
    ]
)

print("Working memory supplied to the model:\n", forensic_memory)
print("\nEvidence-bounded response:\n", forensic_answer)


## What to Notice

- **Memory is selected context.** The agent chooses which earlier information to include in the next model request.
- **Memory is not evidence by itself.** A note can preserve an observed fact, but it does not make an unverified claim true.
- **Bounded memory helps inspection.** Separating observations, unknowns, and next steps makes it easier for a human reviewer to see what the agent relied on.
- **Memory should stay relevant and limited.** Carrying unnecessary or sensitive information into later prompts can reduce clarity and create risk.

Next, open [06_forensic_agent_walkthrough.ipynb](06_forensic_agent_walkthrough.ipynb) to see how role, tools, memory, validation, and boundaries work together in a small agent workflow.